# Module 06: Multiple Series and Layouts

Every figure we have built so far has shown one dataset on one set of axes. Real scientific papers routinely do more: a single panel that overlays two or three related measurements, or a multi-panel figure that presents parallel comparisons side by side. This module covers both, starting with how to put multiple data series on one set of axes and finishing with the layout tools that let you arrange panels precisely.

## Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from cycler import cycler   # standalone package, installed alongside Matplotlib

## Shared datasets

Two synthetic datasets are used throughout this module. The first is a polymer composite series: tensile strength, elongation at break, and Shore D hardness measured as a function of filler content. All three properties are real responses to filler loading and their trends reflect known behavior: tensile strength and hardness increase with filler content while elongation decreases. The second dataset covers a RAFT polymerization experiment where the initiator concentration is varied and we track monomer conversion, number-average molecular weight, and dispersity.

In [ ]:
np.random.seed(42)

# ── Polymer composite dataset ──────────────────────────────────────────────────
# x: filler content in weight percent
filler_wt = np.array([0, 5, 10, 15, 20, 25, 30, 35, 40])

tensile_strength = np.array([28.0, 31.5, 35.2, 38.8, 42.1, 44.9, 46.8, 47.5, 47.9])  # MPa
elongation_break = np.array([420,  370,  310,  255,  200,  155,  115,   85,   62])    # %
shore_d_hardness = np.array([ 55,   58,   62,   65,   68,   70,   72,   74,   75])    # Shore D

# ── RAFT polymerization dataset ───────────────────────────────────────────────
# x: initiator concentration [Init] in mol/L (0.005 to 0.30 mol/L)
n_pts     = 40
init_conc = np.sort(np.random.uniform(0.005, 0.30, n_pts))  # mol/L

# Conversion (X): rises from ~0.1 toward ~0.65, power-law saturation with scatter
conv_true  = 0.68 * (init_conc / 0.30) ** 0.45
conversion = np.clip(conv_true  + np.random.normal(0, 0.025, n_pts), 0.05, 0.80)

# Mn (g/mol): increases from ~420 to ~900 with scatter
Mn_true = 420 + 1000 * (init_conc / 0.30) ** 0.60
Mn      = np.clip(Mn_true + np.random.normal(0, 35, n_pts), 380, 960)

# Dispersity (D-bar): high at low [Init], decaying toward ~1.25 at higher [Init]
D_true    = 1.25 + 1.20 * np.exp(-init_conc / 0.025)
dispersity = np.clip(D_true + np.random.normal(0, 0.04, n_pts), 1.15, 2.60)

## Multiple series on a single axis

The most direct way to compare related measurements is to overlay them on the same set of axes. Each call to `ax.plot()` adds a new series, and Matplotlib automatically cycles through its default color sequence so each series gets a distinct color. A legend identifies the series by the `label` string you attach to each plot call.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(filler_wt, tensile_strength, color='#2166ac', marker='o', linewidth=2, markersize=6, label='Tensile strength (MPa)')
ax.plot(filler_wt, elongation_break, color='#d6604d', marker='s', linewidth=2, markersize=6, label='Elongation at break (%)')
ax.plot(filler_wt, shore_d_hardness, color='#4dac26', marker='^', linewidth=2, markersize=6, label='Shore D hardness')

ax.set_xlabel('Filler content (wt%)', fontsize=12)
ax.set_ylabel('Property value', fontsize=12)
ax.set_title('Three properties on one axis', fontsize=13)
ax.legend(frameon=False, fontsize=10)
ax.grid(which='major', linestyle='--', linewidth=0.5, color='grey', alpha=0.5)

plt.tight_layout()
plt.show()

The plot is technically correct but not particularly useful: elongation at break runs up to 420% while Shore D hardness only reaches 75, so the y axis is stretched to accommodate elongation and the other two curves are compressed into the lower portion of the panel. The shape of the hardness response is almost unreadable, and tensile strength is only marginally better. Sharing a single y axis only works when all series have comparable magnitudes and units; otherwise the scale of one series dominates and the others become illegible.

## Twin-axis plot

`ax.twinx()` creates a second axes object that shares the same x axis but has an independent y axis on the right side. The two axes are overlaid in the same panel and each can be scaled and labeled independently. This solves the mismatched-magnitude problem we just saw with the composite data: elongation (up to 420%) can sit on the left axis while tensile strength (28 to 48 MPa) and Shore D hardness (55 to 75) share a right axis with a much tighter range. The result is that every curve now uses most of its available axis height, and the shape of each response becomes clearly visible.

In [ ]:
color_elong   = '#d6604d'   # red: elongation on the left axis
color_tensile = '#2166ac'   # blue: tensile strength on the right axis
color_shore   = '#4dac26'   # green: Shore D on the right axis

fig, ax_left = plt.subplots(figsize=(8, 5))

# ── Left axis: elongation at break ────────────────────────────────────────────
line1, = ax_left.plot(filler_wt, elongation_break, color=color_elong, linewidth=2, marker='s', markersize=6)
ax_left.set_xlabel('Filler content (wt%)', fontsize=12)
ax_left.set_ylabel('Elongation at break (%)', fontsize=12, color=color_elong)
ax_left.tick_params(axis='y', labelcolor=color_elong)
ax_left.spines['left'].set_color(color_elong)

# ── Right axis: tensile strength and Shore D ──────────────────────────────────
ax_right = ax_left.twinx()
line2, = ax_right.plot(filler_wt, tensile_strength, color=color_tensile, linewidth=2, marker='o', markersize=6)
line3, = ax_right.plot(filler_wt, shore_d_hardness, color=color_shore, linewidth=2, marker='^', markersize=6)
ax_right.set_ylabel('Tensile strength (MPa) / Shore D hardness', fontsize=12, color='gray')
ax_right.tick_params(axis='y', labelcolor='gray')
ax_right.spines['right'].set_color('gray')

# ── Combined legend ───────────────────────────────────────────────────────────
ax_left.legend(
    [line1, line2, line3],
    ['Elongation at break (%)', 'Tensile strength (MPa)', 'Shore D hardness'],
    frameon=False, fontsize=10, loc='center right'
)

ax_left.set_title('Composite properties: twin-axis resolves the scale problem', fontsize=12)
ax_left.grid(which='major', linestyle='--', linewidth=0.5, color='grey', alpha=0.4)
plt.tight_layout()
plt.show()

Elongation now occupies the full height of the left axis and its steep decline is clearly visible. Tensile strength and Shore D share the right axis, which works because their numerical ranges are similar enough that neither overwhelms the other. Coloring each axis label and its tick marks to match the corresponding data series lets the reader match each curve to its scale without reading the legend.

### RAFT polymerization: a case for three independent axes

Sometimes two axes are still not enough. The RAFT dataset has three outputs, each with a fundamentally different range and physical meaning: conversion X runs from 0 to about 0.65 (dimensionless), Mn from roughly 400 to 900 g/mol, and dispersity Đ from about 1.2 up to 2.5 at the lowest initiator concentrations. No pair of these can reasonably share a y axis. Plotting all three together in one panel tells a coherent story: as initiator concentration rises, monomer conversion increases, chains grow longer, and the reaction becomes better controlled as Đ falls toward its limiting value.

In [ ]:
color_conv = '#2166ac'   # blue: conversion on the left axis
color_Mn   = '#d6604d'   # red: Mn on the first right axis
color_D    = '#4dac26'   # green: dispersity on the second right axis (offset outward)

fig, ax1 = plt.subplots(figsize=(8, 5))

# ── ax1: conversion (left axis) ───────────────────────────────────────────────
sc1 = ax1.scatter(init_conc, conversion, color=color_conv, s=30, zorder=3)
ax1.set_xlabel('Initiator concentration [Init] (mol/L)', fontsize=12)
ax1.set_ylabel('Conversion X', fontsize=12, color=color_conv)
ax1.tick_params(axis='y', labelcolor=color_conv)
ax1.spines['left'].set_color(color_conv)
ax1.set_ylim(0, 0.85)

# ── ax2: Mn (first right axis) ────────────────────────────────────────────────
ax2 = ax1.twinx()
sc2 = ax2.scatter(init_conc, Mn, color=color_Mn, s=30, zorder=3)
ax2.set_ylabel('$M_n$ (g/mol)', fontsize=12, color=color_Mn)
ax2.tick_params(axis='y', labelcolor=color_Mn)
ax2.spines['right'].set_color(color_Mn)
ax2.set_ylim(300, 1050)

# ── ax3: dispersity (second right axis, offset outward) ───────────────────────
ax3 = ax1.twinx()
ax3.spines['right'].set_position(('outward', 65))  # move 65 pt outward to clear ax2
ax3.spines['right'].set_color(color_D)
sc3 = ax3.scatter(init_conc, dispersity, color=color_D, s=30, zorder=3)
ax3.set_ylabel('Dispersity Đ (= Mw/Mn)', fontsize=12, color=color_D)
ax3.tick_params(axis='y', labelcolor=color_D)
ax3.set_ylim(1.0, 2.8)

# ── Combined legend ───────────────────────────────────────────────────────────
ax1.legend(
    [sc1, sc2, sc3],
    ['Conversion X', '$M_n$ (g/mol)', 'Dispersity Đ'],
    frameon=True, fontsize=10, loc='center right'
)

ax1.set_title('RAFT polymerization outcomes vs. initiator concentration', fontsize=12)
ax1.grid(which='major', linestyle='--', linewidth=0.5, color='grey', alpha=0.4)

# Extra right margin to accommodate the offset spine and its labels
fig.subplots_adjust(right=0.78)
plt.show()

The second right axis is offset with `ax3.spines['right'].set_position(('outward', 65))`, moving it 65 points to the right of the default position so it does not overlap the Mn axis. `fig.subplots_adjust(right=0.78)` shrinks the plotting area to create space for that extra spine outside the figure boundary. If your tick labels are longer than these, you may need to increase the offset value and decrease the `right` margin correspondingly.

Multiple y axes are legitimate when the series being compared are genuinely linked by the same independent variable and the comparison between them is the scientific point of the figure. They become misleading when used to make two unrelated quantities appear to track each other, or when the axis ranges are chosen to force a visual alignment that the data does not actually support.

## Subplot grids with `plt.subplots(nrows, ncols)`

For data that truly measures different things, separate panels are more honest than shared axes. `plt.subplots(nrows, ncols)` creates a grid of axes and returns them as a 2D NumPy array. You index into that array to access individual panels.

The `sharex` and `sharey` arguments link the axis limits across panels. When all panels share the same independent variable, a shared x axis ensures the reader can compare trends directly without mentally rescaling, and Matplotlib suppresses redundant tick labels on interior panels automatically.

In [ ]:
# Fourth property for the 2x2 grid
impact_strength = np.array([18.0, 16.5, 14.8, 13.0, 11.2, 9.5, 7.9, 6.8, 5.8])  # kJ/m2

In [ ]:
# Dataset and label pairs for the four panels
panel_data = [
    (tensile_strength, 'Tensile strength (MPa)',  '#2166ac', 'o'),
    (elongation_break, 'Elongation at break (%)', '#d6604d', 's'),
    (shore_d_hardness, 'Shore D hardness',         '#4dac26', '^'),
    (impact_strength,  'Impact strength (kJ/m²)', '#8073ac', 'D'),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)

# axes is a 2x2 array; .flat iterates in row-major order
for ax, (y_data, y_label, color, marker) in zip(axes.flat, panel_data):
    ax.plot(filler_wt, y_data,
            color=color, linewidth=2, marker=marker, markersize=6)
    ax.set_ylabel(y_label, fontsize=11)
    ax.grid(which='major', linestyle='--', linewidth=0.5, color='grey', alpha=0.5)
    ax.tick_params(labelsize=10)

# x axis labels only on the bottom row; sharex suppresses them on the top row
for ax in axes[1]:
    ax.set_xlabel('Filler content (wt%)', fontsize=11)

fig.suptitle('Mechanical properties vs. filler content', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

`axes.flat` is a convenient iterator that walks through the 2D array in row-major order, so you can pair each axes object with its corresponding dataset in a single loop. Each panel now has its own y scale, which means elongation at break can span 0 to 420% while hardness spans 55 to 75, with neither series being compressed.

## A 3x2 grid comparing filler types

A common figure structure in materials papers is a matrix of panels where rows correspond to material variants and columns correspond to properties. The layout below shows tensile strength and elongation at break for three filler types: carbon black, glass fiber, and nanoclay. Each filler follows the same general trend but with different magnitudes, which is exactly the kind of comparison that benefits from aligned axes across rows and columns.

In [ ]:
# Synthetic filler-property datasets for three filler types.
# Values reflect representative reinforcement behavior.

filler_types = ['Carbon black', 'Glass fiber', 'Nanoclay']

tensile_by_filler = {
    'Carbon black': np.array([28.0, 31.5, 35.2, 38.8, 42.1, 44.9, 46.8, 47.5, 47.9]),
    'Glass fiber':  np.array([28.0, 34.2, 40.8, 47.5, 54.0, 59.8, 64.2, 67.0, 68.5]),
    'Nanoclay':     np.array([28.0, 30.5, 33.2, 35.8, 37.9, 39.5, 40.6, 41.3, 41.7]),
}

elongation_by_filler = {
    'Carbon black': np.array([420, 370, 310, 255, 200, 155, 115,  85,  62]),
    'Glass fiber':  np.array([420, 340, 265, 205, 155, 115,  82,  58,  40]),
    'Nanoclay':     np.array([420, 385, 345, 305, 268, 232, 198, 168, 142]),
}

filler_colors = {
    'Carbon black': '#2166ac',
    'Glass fiber':  '#d6604d',
    'Nanoclay':     '#4dac26',
}

In [ ]:
# 3 rows (filler types) x 2 columns (properties)
# sharex links all x axes; sharey='col' links y scales within each column only
fig, axes = plt.subplots(
    3, 2,
    figsize=(10, 9),
    sharex=True,
    sharey='col'
)

col_labels = ['Tensile strength (MPa)', 'Elongation at break (%)']
col_data   = [tensile_by_filler, elongation_by_filler]

for row_idx, filler in enumerate(filler_types):
    color = filler_colors[filler]
    for col_idx, (data_dict, col_label) in enumerate(zip(col_data, col_labels)):
        ax = axes[row_idx, col_idx]
        ax.plot(filler_wt, data_dict[filler],
                color=color, linewidth=2, marker='o', markersize=5)
        ax.grid(which='major', linestyle='--', linewidth=0.5, color='grey', alpha=0.5)
        ax.tick_params(labelsize=9)

        # Left column: filler name as a colored y axis label
        if col_idx == 0:
            ax.set_ylabel(filler, fontsize=11, fontweight='bold', color=color)

        # Top row: property name as a column header
        if row_idx == 0:
            ax.set_title(col_label, fontsize=11)

        # Bottom row only: x axis label
        if row_idx == 2:
            ax.set_xlabel('Filler content (wt%)', fontsize=10)

fig.suptitle('Mechanical response to filler loading by filler type',
             fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

`sharey='col'` links y scales within each column but not between columns, which is the right choice here: tensile strength and elongation have different units and numerical ranges. The filler name is used as the y axis label for the left column and colored to match the data, so row identity is always visible without a separate legend. Column headers appear only on the top row and the x axis label only on the bottom row, keeping the figure clean.

## Unequal panel sizes with GridSpec

`plt.subplots()` creates panels of equal size by default. When you need panels of different proportions, `matplotlib.gridspec.GridSpec` gives you explicit control. You define the grid dimensions and assign each axes object to a region of the grid using slice notation, the same way you would index a 2D array.

A common layout in mechanics papers is one wide panel showing a full stress-strain curve on the left and two narrower panels on the right showing key extracted parameters. The large panel shows the complete dataset; the smaller panels highlight the specific quantities being compared.

In [ ]:
# Synthetic stress-strain curves for three filler loadings

def stress_strain_curve(strain, E, sigma_y, hardening, epsilon_break):
    """Simple bilinear stress-strain model with a sharp fracture at epsilon_break."""
    stress = np.where(
        strain <= sigma_y / E,
        E * strain,                                              # elastic region
        sigma_y + hardening * (strain - sigma_y / E)            # plastic region
    )
    stress = np.where(strain <= epsilon_break, stress, np.nan)  # fracture cutoff
    return stress

strain = np.linspace(0, 0.55, 500)

# (Young's modulus MPa, yield stress MPa, hardening MPa, fracture strain, label, color)
curve_params = [
    (1800, 28, 15, 0.50, '0 wt%',  '#2166ac'),
    (2400, 36, 20, 0.35, '15 wt%', '#d6604d'),
    (3100, 43, 22, 0.18, '30 wt%', '#4dac26'),
]

# Extracted parameters for the right-hand bar charts
loadings               = [0, 15, 30]
yield_stress_extracted = [28, 36, 43]   # MPa
elong_extracted        = [50, 35, 18]   # %

In [ ]:
# 2 rows x 3 columns.
# Left panel spans both rows and the first column.
# Middle column is a narrow spacer. Right panels occupy one row each.
fig = plt.figure(figsize=(11, 6))
gs  = gridspec.GridSpec(
    2, 3,
    figure=fig,
    width_ratios=[2, 0.08, 1],   # left panel twice as wide; middle column is a gap
    hspace=0.45,
    wspace=0.08
)

ax_main = fig.add_subplot(gs[:, 0])   # all rows, first column
ax_top  = fig.add_subplot(gs[0, 2])   # row 0, third column
ax_bot  = fig.add_subplot(gs[1, 2])   # row 1, third column

# ── Left panel: full stress-strain curves ─────────────────────────────────────
for E, sigma_y, hardening, epsilon_break, label, color in curve_params:
    stress = stress_strain_curve(strain, E, sigma_y, hardening, epsilon_break)
    ax_main.plot(strain * 100, stress,
                 color=color, linewidth=2.2, label=label)

ax_main.set_xlabel('Engineering strain (%)', fontsize=12)
ax_main.set_ylabel('Engineering stress (MPa)', fontsize=12)
ax_main.set_title('Stress-strain curves', fontsize=12)
ax_main.legend(title='Filler content', frameon=False, fontsize=10, title_fontsize=10)
ax_main.grid(which='major', linestyle='--', linewidth=0.5, color='grey', alpha=0.5)
ax_main.tick_params(labelsize=10)

# ── Upper right panel: yield stress ───────────────────────────────────────────
colors_bar = ['#2166ac', '#d6604d', '#4dac26']

ax_top.bar(loadings, yield_stress_extracted, width=8, color=colors_bar, edgecolor='white')
ax_top.set_ylabel('Yield stress (MPa)', fontsize=10)
ax_top.set_title('Yield stress', fontsize=10)
ax_top.tick_params(labelsize=9)
ax_top.set_xticks(loadings)

# ── Lower right panel: elongation at break ────────────────────────────────────
ax_bot.bar(loadings, elong_extracted, width=8, color=colors_bar, edgecolor='white')
ax_bot.set_xlabel('Filler content (wt%)', fontsize=10)
ax_bot.set_ylabel('Elongation at break (%)', fontsize=10)
ax_bot.set_title('Elongation at break', fontsize=10)
ax_bot.tick_params(labelsize=9)
ax_bot.set_xticks(loadings)

fig.suptitle('Stress-strain analysis: effect of filler content',
             fontsize=14, fontweight='bold', y=1.01)

plt.show()

`GridSpec` accepts `width_ratios` and `height_ratios` as lists that control the relative size of each column and row. The middle column here has a ratio of 0.08, making it a narrow visual gap between the left and right panels. `gs[:, 0]` uses the same NumPy slice syntax you already know: `:` selects all rows and `0` selects the first column, so the left axes spans the full height. The `hspace` and `wspace` arguments control the vertical and horizontal spacing between panels in figure-fraction units.